# ANN с помощью библиотеки faiss

В этом тюториале мы научимся эффективно искать ближайших соседей с помощью библиотеки _faiss_.

Код и документация библиотеки: https://github.com/facebookresearch/faiss

Ее можно поставить в виртуальное окружение командой `pip install faiss-cpu`

Импортируем модули которые нам понадобятся впоследствии:

In [ ]:
import logging
import pathlib
from timeit import default_timer as timer

import faiss
import gensim
import numpy as np

Подготовим логгер:

In [ ]:
logging.basicConfig(level=logging.INFO)

## Данные

В качестве данных будем использовать предобученные эмбеддинги слов _word2vec_.

Домашняя страница: https://code.google.com/archive/p/word2vec/

Ссылка, по которой можно скачать предобученные векторы: https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing

Посморим на уже скачанные данные:

In [ ]:
!ls -lh models/word2vec

Загрузим вектора _word2vec_ с помощью библиотеки _gensim_:

In [ ]:
w2v_model_file = "models/word2vec/GoogleNews-vectors-negative300.bin"

# Load word2vec embeddings
w2v_model = gensim.models.KeyedVectors.load_word2vec_format(w2v_model_file, binary=True)
keys = list(w2v_model.key_to_index.keys())
print(keys[:10])
logging.info(f"loaded embeddings: num_keys = {len(keys)} w2v_model_file = '{w2v_model_file}'")

Получим из загруженной модели матрицу эмбеддингов:

In [ ]:
E = w2v_model[keys]
logging.info(f"extracted embeddings: E.shape = {E.shape}")

## Индексация

Построим индекс _faiss_.

Существует множество разновидностей индексов, самые популярные:
- Flat-индексы, которые позволяют делать полный KNN
- IVF
- HNSW

У каждого индекса существует множество настроек, видов квантизации и т.п.
Подробнее про все эти индексы можно почитать по ссылке: https://github.com/facebookresearch/faiss/wiki/Faiss-indexes

In [ ]:
# Index params
embedding_size = E.shape[1]
nlist = 100                   # number of cells for IVF index
metric = faiss.METRIC_L2      # alternative: faiss.METRIC_INNER_PRODUCT

start = timer()

# Create quantizer and index
quantizer = faiss.IndexFlatIP(embedding_size) # assigns vectors to Voronoi cells
faiss_index = faiss.IndexIVFFlat(quantizer, embedding_size, nlist, metric)

# Compute centroids
faiss_index.train(E)

# Populate index with embeddings
faiss_index.add(E)
logging.info(f"built faiss index: elapsed = {timer() - start:.3f}")

Пре необходимости, можем сохранить индекс в файл:

In [ ]:
tmp_dir = pathlib.Path("tmp")
index_file = tmp_dir.joinpath(f"faiss.index")

# Save index
faiss.write_index(faiss_index, str(index_file))
logging.info(f"saved faiss index: index_file = '{index_file}'")

И загрузить его обратно:

In [ ]:
# Load index
faiss_index = faiss.read_index(str(index_file))
logging.info(f"loaded faiss index: index_file = '{index_file}'")

## Приближенный поиск ближайших соседей

Теперь можно попробовать поискать какое-нибудь слово в нашем индексе.

Тут мы столкнемся с двумя проблемами:
- нам понадобится эмбеддинг слова, которое мы будем искать. Будем считать, что наше слово есть в словаре, т.е. уже известно модели
- _faiss_ оперирует не словами, а ID слов. Эти ID назначаются словам в порядке добавления их в индекс, т.е. по сути являются номерами соответствующих словам строк в исходной матрице эмбеддмнгов 

Построим прямой и обратные словари слово <-> индекс слова:

In [ ]:
# Keyword -> index vocabulary
key2idx = { key: idx for idx, key in enumerate(keys) }

# Keyword -> index reverse vocabulary
idx2key = { idx: key for key, idx in key2idx.items() }
print(list(idx2key.items())[:10])

Возьмем какое-нибудь ключевое слово и достанем из матрицы эмбеддингов его эмбеддинг:

In [ ]:
# Get keyword index
key = "astronomy"
key_idx = key2idx[key]
logging.info(f"keyword found in vocabulary: key = '{key}' key_idx = {key_idx}")

# Get keyword embedding
e_key = E[key_idx]
print(e_key[:10])
logging.info(f"got keyword embedding: key = '{key}' e_key.shape = {e_key.shape}")

Найдем ближайших соседей:

In [ ]:
# Convert keyword embedding to list
E_keys = np.array([e_key])

# How many cells to look in
faiss_index.nprobe = 30

# Search
start = timer() 
D, I = faiss_index.search(E_keys, k=20)
print(I)
print(D)
logging.info(f"got k-NN: D.shape = {D.shape} I.shape = {I.shape} elapsed = {timer() - start:.3f}")

На выходе получаем два массива:
- I -- это индексы найденных ближайших соседей
- D -- это расстояния этих соседей

Превратим индексы в слова с помощью подготовленных ранее словарей:

In [ ]:
# Unpack found indexes and distances
idxs_knn = I[0]
dists_knn = D[0]
keys_knn = [idx2key[idx] for idx in idxs_knn]

# Print results
print(f"QUERY: key = {key}")
for i, key_knn in enumerate(keys_knn):
    dist = dists_knn[i]
    print(f"\tKNN: i = {i} key_knn = {key_knn} dist = {dist:.3f}")

Видим, что мы нашли слова, которые по смыслу близки к нашему ключевому слову.

Таким образом, мы:
- познакомились с библиотекой _faiss_
- научились использовать ее для приближенного поиска ближайших соседей

Дальше самостоятельно вы можете:
- попробовать разные виды индексов, и поиграться с их параметрами
- посчитать метрики качества, напр. какую долю из настоящего числа ближайших соседей находит ANN